# Manim Studio — Session 01

Technical visualizations for *Data, Representations, and the First Model* (Manim Community Edition).

**Use:** run the setup cell once, then run a scene cell to preview inline. Run the export cell at the end to copy clean MP4s into `media/session_01/`, then paste the printed `<video>` snippet into `sessions/session_01.md`.


In [ ]:
# Setup (run once per kernel). First time only:  pip install manim   (+ ffmpeg)
from manim import *
config.media_dir = 'media'
config.background_color = '#0f172a'
import manim; print('Manim', manim.__version__)


### 1 · Perceptron learns the decision boundary
The learning rule nudges the line after each misclassified point until the two classes are separated.

In [ ]:
%%manim -qm -v WARNING PerceptronLearns
from manim import *
import numpy as np

class PerceptronLearns(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'
        ax = Axes(x_range=[-3, 3, 1], y_range=[-3, 3, 1], x_length=6.5, y_length=6.5,
                  axis_config={'include_tip': False, 'stroke_opacity': 0.4})
        rng = np.random.default_rng(2)
        A = rng.normal([1.3, 1.3], 0.45, size=(8, 2))   # class 1 (blue)
        B = rng.normal([-1.3, -1.3], 0.45, size=(8, 2)) # class 0 (yellow)
        X = np.vstack([A, B]); Y = np.array([1]*8 + [0]*8)
        dA = VGroup(*[Dot(ax.c2p(*p), color=BLUE_B, radius=0.08) for p in A])
        dB = VGroup(*[Dot(ax.c2p(*p), color=YELLOW, radius=0.08) for p in B])
        title = Text('Perceptron learning rule', font_size=26).to_edge(UP, buff=0.3)
        self.play(Create(ax), Write(title))
        self.play(FadeIn(dA), FadeIn(dB))

        def boundary(w, b):
            pts = []
            for x in (-3, 3):
                if abs(w[1]) > 1e-9:
                    yy = -(w[0]*x + b)/w[1]
                    if -3 <= yy <= 3: pts.append((x, yy))
            for yv in (-3, 3):
                if abs(w[0]) > 1e-9:
                    xx = -(w[1]*yv + b)/w[0]
                    if -3 <= xx <= 3: pts.append((xx, yv))
            uniq = []
            for p in pts:
                if all(abs(p[0]-q[0]) + abs(p[1]-q[1]) > 1e-6 for q in uniq): uniq.append(p)
            if len(uniq) < 2: uniq = [(-3, 0), (3, 0)]
            return Line(ax.c2p(*uniq[0]), ax.c2p(*uniq[1]), color=GREEN_B, stroke_width=4)

        w = np.array([-1.5, 2.0]); b = 0.5; eta = 0.2
        line = boundary(w, b)
        self.play(Create(line))
        changed, epoch = True, 0
        while changed and epoch < 6:
            changed, epoch = False, epoch + 1
            for xi, yi in zip(X, Y):
                pred = 1 if (w @ xi + b) >= 0 else 0
                if pred != yi:
                    w = w + eta*(yi - pred)*xi
                    b = b + eta*(yi - pred)
                    changed = True
                    self.play(Transform(line, boundary(w, b)), run_time=0.4)
        self.wait(1.5)


### 2 · XOR: a hidden layer warps the space
The whole input grid is bent by the feature map (x+y, (x−y)²). After the warp, a single straight line separates the XOR classes.

In [ ]:
%%manim -qm -v WARNING XORWarp
from manim import *
import numpy as np

class XORWarp(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'
        O = np.array([0.0, -1.3, 0.0]); S = 1.0
        def Cc(x, y): return O + np.array([x*S, y*S, 0.0])
        def inv(p): return ((p[0]-O[0])/S, (p[1]-O[1])/S)
        def g(x, y): return (x + y, (x - y)**2)
        F = lambda p: Cc(*g(*inv(p)))

        title = Text('A hidden layer warps the input space', font_size=24).to_edge(UP, buff=0.3)
        grid = VGroup()
        vals = np.linspace(-1, 1, 5); samp = np.linspace(-1, 1, 30)
        for x in vals:
            ln = VMobject().set_points_as_corners([Cc(x, t) for t in samp]); grid.add(ln)
        for y in vals:
            ln = VMobject().set_points_as_corners([Cc(t, y) for t in samp]); grid.add(ln)
        grid.set_stroke(BLUE_D, width=2, opacity=0.45)

        P = [((-0.5, -0.5), BLUE_B), ((0.5, 0.5), BLUE_B),
             ((-0.5, 0.5), YELLOW), ((0.5, -0.5), YELLOW)]
        dots = VGroup(*[Dot(Cc(x, y), color=c, radius=0.12) for (x, y), c in P])
        self.play(Write(title), Create(grid), FadeIn(dots))
        self.wait(0.5)

        targets = [Cc(*g(x, y)) for (x, y), c in P]
        self.play(grid.animate.apply_function(F),
                  *[d.animate.move_to(t) for d, t in zip(dots, targets)], run_time=3)
        sep = Line(np.array([O[0]-2.3, O[1]+0.5*S, 0]), np.array([O[0]+2.3, O[1]+0.5*S, 0]),
                   color=GREEN_B, stroke_width=4)
        ok = Text('now one straight line separates the classes', font_size=20, color=GREEN_B).to_edge(DOWN, buff=0.3)
        self.play(Create(sep), Write(ok)); self.wait(2)


### 3 · Activation functions and their derivatives
Solid = function, dashed = derivative. Note how sigmoid/tanh derivatives vanish at the extremes, while ReLU's stays 1.

In [ ]:
%%manim -qm -v WARNING ActivationFunctions
from manim import *
import numpy as np

class ActivationFunctions(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'
        sig = lambda x: 1/(1 + np.exp(-x))
        funcs = [
            ('Step',       lambda x: 1.0 if x >= 0 else 0.0, None),
            ('Sigmoid',    sig, lambda x: sig(x)*(1 - sig(x))),
            ('Tanh',       lambda x: np.tanh(x), lambda x: 1 - np.tanh(x)**2),
            ('ReLU',       lambda x: max(0.0, x), lambda x: 1.0 if x > 0 else 0.0),
            ('Leaky ReLU', lambda x: x if x > 0 else 0.1*x, lambda x: 1.0 if x > 0 else 0.1),
            ('GELU',       lambda x: x*sig(1.702*x), None),
        ]
        cells = VGroup()
        for name, f, df in funcs:
            ax = Axes(x_range=[-2.5, 2.5, 1], y_range=[-1.3, 2.8, 1], x_length=3.4, y_length=2.4,
                      axis_config={'include_tip': False, 'stroke_opacity': 0.4, 'stroke_width': 2})
            grp = VGroup(ax, ax.plot(f, x_range=[-2.5, 2.5], color=BLUE_B, use_smoothing=False))
            if df is not None:
                grp.add(DashedVMobject(ax.plot(df, x_range=[-2.5, 2.5], color=YELLOW, use_smoothing=False), num_dashes=40))
            grp.add(Text(name, font_size=20).next_to(ax, UP, buff=0.1))
            cells.add(grp)
        cells.arrange_in_grid(rows=2, cols=3, buff=0.7)
        cells.scale_to_fit_width(12.6).to_edge(UP, buff=0.6)
        self.play(LaggedStart(*[FadeIn(c) for c in cells], lag_ratio=0.2, run_time=4))
        legend = VGroup(Text('— function', font_size=18, color=BLUE_B),
                        Text('-- derivative', font_size=18, color=YELLOW)).arrange(RIGHT, buff=0.7).to_edge(DOWN, buff=0.25)
        self.play(FadeIn(legend)); self.wait(2)


### 4 · Standardization conditions the loss surface
Features on very different scales give elongated loss contours (slow descent). After standardization the contours are round and gradient descent converges directly.

In [ ]:
%%manim -qm -v WARNING Standardization
from manim import *
import numpy as np

class Standardization(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'
        rng = np.random.default_rng(0)
        pts = rng.normal(0, 1, size=(60, 2)) * np.array([2.2, 0.5])
        axL = Axes(x_range=[-6, 6, 2], y_range=[-3, 3, 1], x_length=6, y_length=4,
                   axis_config={'include_tip': False, 'stroke_opacity': 0.4}).to_edge(LEFT, buff=0.5).shift(DOWN*0.2)
        cloud = VGroup(*[Dot(axL.c2p(*p), color=BLUE_B, radius=0.05) for p in pts])
        ell = VGroup(*[Ellipse(width=w*2.0, height=w*0.7, color=YELLOW, stroke_opacity=0.5).move_to(axL.c2p(0, 0)) for w in (1, 2, 3)])
        tl = Text('raw: different scales', font_size=20).next_to(axL, UP, buff=0.15)
        self.play(Create(axL), FadeIn(cloud), Create(ell), Write(tl)); self.wait(1)

        axR = Axes(x_range=[-3, 3, 1], y_range=[-3, 3, 1], x_length=4.4, y_length=4.4,
                   axis_config={'include_tip': False, 'stroke_opacity': 0.4}).to_edge(RIGHT, buff=0.5).shift(DOWN*0.2)
        z = (pts - pts.mean(0)) / pts.std(0)
        cloudZ = VGroup(*[Dot(axR.c2p(*p), color=BLUE_B, radius=0.05) for p in z])
        circ = VGroup(*[Circle(radius=r, color=GREEN_B, stroke_opacity=0.6).move_to(axR.c2p(0, 0)) for r in (0.8, 1.6, 2.4)])
        tr = Text('standardized: unit variance', font_size=18).next_to(axR, UP, buff=0.15)
        self.play(Create(axR)); self.play(TransformFromCopy(cloud, cloudZ), Write(tr)); self.play(Create(circ))
        cap = Text('round contours -> faster gradient descent', font_size=20, color=GREEN_B).to_edge(DOWN, buff=0.25)
        self.play(Write(cap)); self.wait(2)


### Export & embed
Run scenes with `-qh` for the final 1080p render, then run the cell below. It copies the latest MP4s into `media/session_01/` and prints ready-to-paste `<video>` blocks for `sessions/session_01.md`.

In [ ]:
import glob, os, shutil

EXPORTS = {
    'PerceptronLearns':    'perceptron_boundary',
    'XORWarp':             'xor_warp',
    'ActivationFunctions': 'activation_functions',
    'Standardization':     'standardization',
}
os.makedirs('media/session_01', exist_ok=True)
for scene, name in EXPORTS.items():
    hits = glob.glob(f'media/videos/**/{scene}*.mp4', recursive=True)
    if not hits:
        print(f'(no render yet for {scene})'); continue
    src = max(hits, key=os.path.getmtime)
    dst = f'media/session_01/{name}.mp4'
    shutil.copy(src, dst)
    print(f'saved {dst}')
    print(f'  <video src="media/session_01/{name}.mp4" controls loop muted playsinline style="max-width:100%;border-radius:10px;"></video>')
